# Vincimap - lancement Colab

Notebook pour lancer `main.py` depuis Google Colab. Avant d'executer, active un runtime GPU dans Colab : `Runtime > Change runtime type > GPU`.

Le notebook separe les trois actions du script : `create_workspace`, `run_colmap`, puis `run_gaussian_training`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuration

Renseigne `PROJECT_DIR` si le repo est deja dans Google Drive. Sinon, mets l'URL Git dans `REPO_URL` et laisse `PROJECT_DIR = '/content/vincimap'`.

In [ ]:
from pathlib import Path
import os

# Option 1: repo deja copie dans Google Drive.
# PROJECT_DIR = '/content/drive/MyDrive/vincimap'

# Option 2: clone depuis Git. Laisse REPO_URL vide si tu utilises l'option 1.
REPO_URL = ''
PROJECT_DIR = '/content/vincimap'

# Fichiers d'entree dans Drive.
VIDEO_PATH = '/content/drive/MyDrive/vincimap_inputs/video.mp4'
DISTANCES_PATH = '/content/drive/MyDrive/vincimap_inputs/distances.txt'

# Workspace de sortie. Le mettre dans Drive conserve les resultats apres la session Colab.
WORKSPACE_PATH = '/content/drive/MyDrive/vincimap_workspaces/video'

if REPO_URL and not Path(PROJECT_DIR).exists():
    !git clone {REPO_URL} {PROJECT_DIR}

project_path = Path(PROJECT_DIR)
assert project_path.exists(), f'PROJECT_DIR introuvable: {PROJECT_DIR}'
assert Path(VIDEO_PATH).exists(), f'VIDEO_PATH introuvable: {VIDEO_PATH}'
assert Path(DISTANCES_PATH).exists(), f'DISTANCES_PATH introuvable: {DISTANCES_PATH}'

os.chdir(project_path)
print('Projet:', Path.cwd())
print('Workspace:', WORKSPACE_PATH)

## Dependances systeme

In [ ]:
!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq ffmpeg colmap
!ffmpeg -version | head -n 1
!colmap -h | head -n 5

## Dependances Python

Cette cellule suit l'ordre d'installation du README, puis installe le reste de `requirements.txt` en excluant les paquets deja installes avec leurs index CUDA specifiques.

In [ ]:
!python -m pip install -q --upgrade pip setuptools wheel
!python -m pip install -q torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1 --index-url https://download.pytorch.org/whl/cu124
!python -m pip install -q ninja numpy jaxtyping rich
!python -m pip install -q gsplat --index-url https://docs.gsplat.studio/whl/pt24cu124

from pathlib import Path

requirements = Path('requirements.txt').read_text(encoding='utf-8').splitlines()
skip_prefixes = ('torch==', 'torchvision==', 'torchaudio==', 'gsplat==')
filtered = [line for line in requirements if not line.strip().startswith(skip_prefixes)]
Path('/tmp/vincimap_requirements_colab.txt').write_text('\n'.join(filtered) + '\n', encoding='utf-8')

!python -m pip install --no-build-isolation -r /tmp/vincimap_requirements_colab.txt

## Verification runtime

In [ ]:
import torch
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
!python main.py --help

## 1. Creation du workspace

Cette etape copie la video et le fichier distances, cree les configs, extrait les frames et lie les distances aux images.

In [ ]:
!python main.py \
  --action create_workspace \
  --video_path "{VIDEO_PATH}" \
  --distances_path "{DISTANCES_PATH}" \
  --workspace_path "{WORKSPACE_PATH}" \
  --colmap_path colmap

## 2. Reconstruction COLMAP

Lance `feature_extractor`, `sequential_matcher`, `mapper`, `point_filtering`, `bundle_adjuster`, `image_undistorter`, puis les conversions de modele.

In [ ]:
!python main.py \
  --action run_colmap \
  --workspace_path "{WORKSPACE_PATH}"

## 3. Training Gaussian

In [ ]:
!python main.py \
  --action run_gaussian_training \
  --workspace_path "{WORKSPACE_PATH}"

## Recuperer les sorties

In [ ]:
!find "{WORKSPACE_PATH}" -maxdepth 3 -type f | sort | head -n 80